# SwarmSort Gen 2, notebook 6: final test-set evaluation of Arms A, B and C

**What:** scores the three Gen 2 arms (A: defaults, B: random search, C: PSO) on the held-out test split with
`src/evaluate.py --arms configs/gen2/arms.yaml`: once on the whole test split (tag `all`) and once per domain
(`studio`, `real_world`, and `india` if the build has it). Each run gives mAP@50 and mAP@50-95 with 95% paired-bootstrap
confidence intervals, pairwise differences, per-class AP, confusion matrices, latency and sample predictions.

**Why:** this is the final, fair comparison. **The test split is used exactly once, here**: run this notebook only
after all three arms are trained, and do not tune anything on its numbers.

**Inputs** (Add Input -> Notebook Output, the finished version of each):

| Notebook output | Gives |
|---|---|
| `g2_0_build_dataset` | the test images |
| `g2_1_arm_a` | `gen2_arm_a_best.pt`, `arm_a.json`, `arm_a_epochs.csv` |
| `g2_4_arm_b` | `gen2_arm_b_best.pt`, `arm_b.json`, `arm_b_epochs.csv`, `best_random.json` |
| `g2_5_arm_c` | `gen2_arm_c_best.pt`, `arm_c.json`, `arm_c_epochs.csv`, `best_pso.json` |

The locating cell finds these files anywhere under `/kaggle/input` and copies them to the paths
`configs/gen2/arms.yaml` expects. **Accelerator: GPU T4 x2**, **Internet: on**. Run it with
**Save Version -> Save & Run All (Commit)**.

**Expected time:** about 30-60 minutes (4 evaluation runs; each validates the three models, times them and draws 1000
bootstrap resamples; Gen 1's single run took about 5 minutes for 1046 test images).

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
assert os.path.isdir(f"{REPO}/src"), "git clone failed: turn on Settings -> Internet and run again"
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4 x2 and run again"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))


def run(*args):
    """Run a repository script. check=True: if the script stops with an error, this cell fails and the version
    is marked failed."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    subprocess.run([sys.executable, *args], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# Find the dataset built by g2_0_build_dataset (attached as input) and write configs that point at it:
# /kaggle/working/gen2_cfg/data.yaml, data_test_<domain>.yaml, split.csv and lists/ (the images stay read-only)
import glob
INPUT, CFG = "/kaggle/input", "/kaggle/working/gen2_cfg"
builds = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT}/**/build_report.json", recursive=True)
                 if os.path.isfile(os.path.join(os.path.dirname(p), "data.yaml"))
                 and os.path.isdir(os.path.join(os.path.dirname(p), "images"))})
if not builds:
    raise FileNotFoundError("The Generation 2 dataset is not attached. Add Input -> Notebook Output -> "
                            "g2_0_build_dataset (its finished version), then run again.")
if len(builds) > 1:
    raise RuntimeError("Several built datasets are attached; remove all but one version of g2_0_build_dataset:\n  "
                       + "\n  ".join(builds))
BUILD = builds[0]
print("Built dataset:", BUILD)
if "--relocate-from" not in open("src/build_dataset.py", encoding="utf-8").read():
    raise RuntimeError("src/build_dataset.py on GitHub has no --relocate-from mode yet: push the latest commits")
run("src/build_dataset.py", "--relocate-from", BUILD, "--out", CFG)
assert os.path.isfile(f"{CFG}/data.yaml"), f"the relocation did not write {CFG}/data.yaml"
print(open(f"{CFG}/data.yaml").read())
print("Test-set yamls:", sorted(os.path.basename(p) for p in glob.glob(f"{CFG}/data_test_*.yaml")))


In [ ]:
# Find each arm's weights, training JSON and epochs CSV in the attached outputs, and copy them to the paths in
# configs/gen2/arms.yaml (models at results/gen2/runs/arm_<x>/weights/best.pt, JSONs and CSVs in results/gen2/)
import hashlib, shutil, yaml
from collections import defaultdict
ARMS_YAML = "configs/gen2/arms.yaml"
arms = yaml.safe_load(open(ARMS_YAML))["arms"]
NOTEBOOK = {"A": "g2_1_arm_a", "B": "g2_4_arm_b", "C": "g2_5_arm_c"}
index = defaultdict(list)  # file name -> every path under /kaggle/input (image and label folders skipped)
for root, dirs, files in os.walk(INPUT):
    dirs[:] = [d for d in dirs if d not in ("images", "labels")]
    for f in files:
        index[f].append(os.path.join(root, f).replace(os.sep, "/"))


def sha(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()


def closest(paths, anchor):
    """The path sharing the longest folder prefix with anchor, i.e. from the same attached output."""
    return max(paths, key=lambda p: (len(os.path.commonpath([p, anchor])), p))


used = []
for label, arm in arms.items():
    x = label.lower()
    nb = NOTEBOOK.get(label, "the notebook that trained arm " + label)
    weights = index[f"gen2_arm_{x}_best.pt"] + [p for p in index["best.pt"]
                                                  if p.endswith(f"results/gen2/runs/arm_{x}/weights/best.pt")]
    if not weights:
        raise FileNotFoundError(f"No weights for arm {label} (gen2_arm_{x}_best.pt) under {INPUT}: "
                                f"Add Input -> Notebook Output -> {nb} (its finished version), then run again.")
    if len({sha(p) for p in weights}) > 1:
        raise RuntimeError(f"Different weights for arm {label} are attached (several versions of {nb}?); keep one:\n  "
                           + "\n  ".join(weights))
    anchor = sorted(weights, key=len)[0]  # the top-level copy if present
    os.makedirs(os.path.dirname(arm["model"]), exist_ok=True)
    shutil.copyfile(anchor, arm["model"])
    used.append((label, "model", anchor, arm["model"]))
    for key in ("val_json", "epochs_csv", "config"):
        dest = arm.get(key)
        if not dest or any(u[3] == dest for u in used):
            continue  # arm A's config is its val_json, already copied
        name = os.path.basename(dest)
        found = [p for p in index[name] if p.endswith(f"results/gen2/{name}")]
        committed = os.path.exists(dest)
        if key == "config" and committed:  # a search winner committed to the repository is authoritative
            used.append((label, key, "(repository)", dest))
        elif found:  # the copy from the same output as the weights belongs to them
            src = closest(found, anchor)
            if committed and sha(src) != sha(dest):
                print(f"Note: {dest} differs from the committed file; using {src}, which came with the weights")
            shutil.copyfile(src, dest)
            used.append((label, key, src, dest))
        elif committed:
            used.append((label, key, "(repository)", dest))
        else:
            raise FileNotFoundError(f"{name} for arm {label} not found in the repository or under {INPUT}: attach "
                                    f"the finished version of {nb}, or commit {dest} and push.")
print(f"{'arm':<4}{'file':<11}{'copied to':<42}from")
for label, key, src, dest in used:
    print(f"{label:<4}{key:<11}{dest:<42}{src}")


In [ ]:
# The test split, scored once: the whole test set (tag all), then each domain. evaluate.py checks every checkpoint
# (7 classes, 100 epochs, imgsz 640 and its arm's hyperparameters) before it validates anything.
DOMAINS = ["studio", "real_world", "india"]
TAGS = []
for tag, data in [("all", f"{CFG}/data.yaml")] + [(d, f"{CFG}/data_test_{d}.yaml") for d in DOMAINS]:
    if not os.path.exists(data):
        if tag == "india":
            print("No India test set in this build (DWSD was not attached to g2_0_build_dataset): india skipped")
            continue
        raise FileNotFoundError(f"{data} missing: the build has no {tag} test set (see g2_0_build_dataset)")
    print(f"\n=== test split, {tag} ===")
    run("src/evaluate.py", "--arms", ARMS_YAML, "--data", data, "--split", "test", "--out-dir", "results/gen2",
        "--tag", tag, "--device", 0)
    TAGS.append(tag)


In [ ]:
# The comparison reports written by evaluate.py
from IPython.display import Markdown, display
for tag in TAGS:
    display(Markdown(open(f"results/gen2/test_{tag}_comparison.md", encoding="utf-8").read()))


In [ ]:
# One table: test mAP@50 [95% bootstrap CI] per arm and domain; also saved as results/gen2/test_domains.md
import json
reports = {tag: json.load(open(f"results/gen2/test_{tag}_results.json")) for tag in TAGS}
header = "| Arm | " + " | ".join(f"{t} ({reports[t]['images']} images)" for t in TAGS) + " |"
lines = ["## Test mAP@50 [95% CI] per domain", "", header, "|---|" + "---|" * len(TAGS)]
for label, arm in arms.items():
    cells = []
    for t in TAGS:
        m = reports[t]["models"][label]
        lo, hi = m["bootstrap"]["map50_ci95"]
        cells.append(f"{m['metrics']['map50']:.3f} [{lo:.3f}, {hi:.3f}]")
    lines.append(f"| {label}: {arm['name']} | " + " | ".join(cells) + " |")
lines += ["", "Pairwise verdicts (paired bootstrap, 95%):"]
for t in TAGS:
    for p in reports[t]["pairwise"]:
        lines.append(f"- {t}, {p['b']} vs {p['a']}: {p['verdict']}")
table = "\n".join(lines) + "\n"
open("results/gen2/test_domains.md", "w", encoding="utf-8").write(table)
print(table)


### Files to download / where they go locally

From the **Output** tab of the finished version (all under `SwarmSort/results/gen2/`):

| File in the Output tab | Where it goes locally |
|---|---|
| `test_<tag>_results.json` (tags all, studio, real_world, india) | `results/gen2/` |
| `test_<tag>_comparison.md` | `results/gen2/` |
| `test_domains.md` (the combined arm x domain table) | `results/gen2/` |
| `plots/` (confusion matrices, per-class AP, samples, training curves) | `results/gen2/plots/` |

Commit them all. The copied weights, JSONs and the `runs/` folder are not needed (the JSONs and CSVs already came from
notebooks 1, 4 and 5).

### If the session dies

Run it again with the same inputs: the evaluation is deterministic (seeded bootstrap), so it gives the same numbers.
It still counts as the single use of the test split.